# Tenglish Intent Classifier: Fine-tuning with Unsloth + LoRA

**Goal:** classify short Tenglish (Telugu in English letters) messages into one of 5 intents: `greeting`, `query`, `complaint`, `request`, `thanks`.

**The experiment:**
1. Measure the **base model** on a held-out test set (baseline)
2. Fine-tune with LoRA
3. Measure again on the **same** test set
4. Compare, look at errors, export


## 1. Setup

In [1]:
%%capture
!pip install unsloth
# Unsloth is a library that makes LLM fine-tuning and faster on supported GPUs.


In [2]:
!nvidia-smi
# Command to check your GPU environment

Mon Nov 16 16:27:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload the dataset
Upload `tenglish_intents.jsonl` when prompted. Each line looks like: `{"text": "...", "label": "..."}`

In [3]:
from google.colab import files
uploaded = files.upload()
# To upload prepared dataset. That itself provides instructions to model.

Saving tenglish_intents.jsonl to tenglish_intents.jsonl


In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
# Data cleaning and Data preparation
df = pd.read_json("tenglish_intents.jsonl", lines=True)
df["text"] = df["text"].str.strip()
df = df.drop_duplicates(subset="text").reset_index(drop=True)
LABELS = sorted(df.label.unique())
print("Rows:", len(df), "| Labels:", LABELS)
print(df.label.value_counts())

# Held-out test set. Never train on this.
train_df, test_df = train_test_split(df, test_size=0.25, stratify=df.label, random_state=42)
print("\nTrain:", len(train_df), "| Test:", len(test_df))

Rows: 100 | Labels: ['complaint', 'greeting', 'query', 'request', 'thanks']
label
greeting     20
query        20
complaint    20
request      20
thanks       20
Name: count, dtype: int64

Train: 75 | Test: 25


## 3. Prompt format
The **same prompt** is used for the baseline and the fine-tuned model, so the comparison is fair.

In [5]:
SYSTEM = (
    "Classify the intent of the user's Tenglish message. "
    "Reply with exactly one label from: " + ", ".join(LABELS) + "."
)
# This creates the instruction that model must follow
def build_messages(text, label=None):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": text},
    ]
    if label is not None:
        msgs.append({"role": "assistant", "content": label})
    return msgs

## 4. Load the model (4-bit) and attach LoRA adapters
`Qwen2.5-1.5B-Instruct` is small enough for a free T4. Right after attaching, the LoRA weights are zero-initialized, so the model behaves exactly like the base model. That gives us a clean baseline.

In [6]:
from unsloth import FastLanguageModel
import torch
# This is to load pretrained model (Qwen 2.5)
MODEL = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_LEN = 256

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL,
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
    dtype=None,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1116 16:28:23.998000 395 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1116 16:28:24.038000 395 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Unsloth: xformers is installed but cannot load with torch 2.14.1+cu130, so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.5: Fast Qwen2 patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Unsloth 2026.11.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Unsloth: compiling each transformer block as one region (Qwen2DecoderLayer).


## 5. Evaluation helpers

In [7]:
from sklearn.metrics import classification_report, confusion_matrix
# Evaluation functions
def predict(texts):
    FastLanguageModel.for_inference(model)
    # Switch model to inference mode
    outs = []
    for t in texts:
        prompt = tokenizer.apply_chat_template(build_messages(t), tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=8, do_sample=False)
        outs.append(tokenizer.decode(gen[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip().lower())
    return outs

def normalize(raw):
    for l in LABELS:
        if raw.startswith(l):
            return l
    return "invalid"   # model did not answer with a valid label

def evaluate(name):
    raw = predict(test_df.text.tolist())
    preds = [normalize(r) for r in raw]
    acc = sum(p == y for p, y in zip(preds, test_df.label)) / len(test_df)
    invalid = sum(p == "invalid" for p in preds)
    print(f"[{name}] accuracy: {acc:.1%} | invalid outputs: {invalid}/{len(preds)}")
    return preds, acc

## 6. Baseline: the model BEFORE fine-tuning

In [8]:
base_preds, base_acc = evaluate("BASE")

Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[BASE] accuracy: 36.0% | invalid outputs: 0/25


## 7. Prepare training data
Training text is the chat template applied to (system, user, assistant=label). Loss is computed only on the assistant's answer.

In [9]:
from datasets import Dataset

def to_text(text, label):
    return tokenizer.apply_chat_template(build_messages(text, label), tokenize=False)

train_ds = Dataset.from_dict({
    "text": [to_text(t, l) for t, l in zip(train_df.text, train_df.label)]
})
print(train_ds[0]["text"])

<|im_start|>system
Classify the intent of the user's Tenglish message. Reply with exactly one label from: complaint, greeting, query, request, thanks.<|im_end|>
<|im_start|>user
Naa plan upgrade cheyyandi<|im_end|>
<|im_start|>assistant
request<|im_end|>



## 8. Train
With a small dataset, 3 epochs is a reasonable start. If the training loss goes near zero while test accuracy stays low, you are overfitting: add more data or reduce epochs.

In [10]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

FastLanguageModel.for_training(model)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_LEN,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

# Only learn from the assistant's reply (the label)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/75 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=2):   0%|          | 0/75 [00:00<?, ? examples/s]

Unsloth: Packing is on by default - documents are packed into full rows with no cross-document attention. Set packing = False in SFTConfig (or UNSLOTH_DEFAULT_PACKING=0) to turn it off.


Map:   0%|          | 0/20 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20 | Num Epochs = 3 | Total steps = 9
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 2 x 1) = 8
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: packed / padding-free rows run each transformer block uncompiled; only unpacked batches use the compiled blocks.


Step,Training Loss
5,0.770969


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-9/tokenizer_config.json.


TrainOutput(global_step=9, training_loss=0.5141624278492398, metrics={'train_runtime': 11.4995, 'train_samples_per_second': 5.218, 'train_steps_per_second': 0.783, 'total_flos': 99859724006400.0, 'train_loss': 0.5141624278492398, 'epoch': 3.0})

## 9. Evaluate AFTER fine-tuning (same test set)

In [11]:
ft_preds, ft_acc = evaluate("FINE-TUNED")
print(f"\nBase: {base_acc:.1%}  ->  Fine-tuned: {ft_acc:.1%}  (change: {ft_acc - base_acc:+.1%})")
print()
print(classification_report(test_df.label, ft_preds, zero_division=0))

Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[FINE-TUNED] accuracy: 88.0% | invalid outputs: 1/25

Base: 36.0%  ->  Fine-tuned: 88.0%  (change: +52.0%)

              precision    recall  f1-score   support

   complaint       1.00      1.00      1.00         5
    greeting       1.00      1.00      1.00         5
     invalid       0.00      0.00      0.00         0
       query       0.83      1.00      0.91         5
     request       0.75      0.60      0.67         5
      thanks       1.00      0.80      0.89         5

    accuracy                           0.88        25
   macro avg       0.76      0.73      0.74        25
weighted avg       0.92      0.88      0.89        25



### Look at the mistakes (this is where you learn the most)

In [12]:
errors = test_df.assign(base=base_preds, tuned=ft_preds)
errors = errors[errors.tuned != errors.label]
pd.set_option("display.max_colwidth", 100)
errors[["text", "label", "base", "tuned"]]

,text,label,base,tuned
73,Ee message ni Telugu loki translate cheyyandi,request,query,invalid
62,Naaku invoice mail cheyyagalara?,request,query,query
93,"Chala bagundi, keep it up",thanks,query,request


## 10. Try your own messages

In [13]:
tests = [
    "Naaku oka help kavali, account lo problem undi",
    "Mee delivery chala late ayindi, em jarugutundi?",
    "Thanks andi, super service",
    "Hello andi, ela unnaru",
    "Ee product eppudu stock lo untundi?",
]
for t, p in zip(tests, predict(tests)):
    print(f"{t}  ->  {normalize(p)}")

Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=8) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Naaku oka help kavali, account lo problem undi  ->  complaint
Mee delivery chala late ayindi, em jarugutundi?  ->  complaint
Thanks andi, super service  ->  thanks
Hello andi, ela unnaru  ->  greeting
Ee product eppudu stock lo untundi?  ->  query


## 11. Save the adapter
This downloads a small zip (the LoRA adapter only, not the full model).

In [ ]:
import shutil
model.save_pretrained("tenglish_lora")
tokenizer.save_pretrained("tenglish_lora")
shutil.make_archive("tenglish_lora", "zip", "tenglish_lora")
files.download("tenglish_lora.zip")

## 12. (Optional) Export for Ollama
Merges the adapter and exports a quantized GGUF file you can run locally with Ollama or llama.cpp. Uncomment to run.

In [ ]:
# model.save_pretrained_gguf("tenglish_gguf", tokenizer, quantization_method="q4_k_m")
# !ls tenglish_gguf

## Next steps
- **Grow the dataset to 500+ rows.** The seed file has 100, which is enough to see the pipeline work but not enough for reliable numbers.
- **Write the test set yourself.** Use real messages from friends and family (with permission), including different spellings ("ledu", "ledhu", "lEdu").
- **Add hard cases:** mixed intents ("thanks, but my refund is still pending"), sarcasm, very short messages.
- **Try a bigger base model** (e.g. 3B or 7B-8B) and compare accuracy against the extra training time.
- **Compare with a hosted LLM** using a good prompt on the same test set, and note accuracy and cost per 1,000 messages.